# 08_screen_libraries — 천연물 3개 DB 스크리닝 (FooDB·COCONUT·NPASS)

**한 줄 요약:** 21에서 만든 **fingerprint 앙상블**로 3개 천연물 DB의 화합물을 예측해 HSD17B13 저해 후보(활성 확률 높은 것)를 뽑는다.
**이전과 차이:** NPASS 1개 → **FooDB·COCONUT·NPASS 3개** 통합. 대형 DB(COCONUT ~70만)라 **청크(chunk)로 나눠** 처리.
**주의:** 배포모델은 지문만 쓴다(2D/3D descriptor는 70만 개엔 비현실적). 점수는 decoy 학습 특성상 부풀 수 있어 **1차 필터**로만.
**큰 흐름:** ① 준비 → ② 모델·지문계산기 → ③ DB 목록 → ④ 청크 스크리닝 → ⑤ 저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
지문 계산·모델 로드에 필요한 라이브러리를 가져온다.

In [ ]:
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())
import time, pickle
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Avalon import pyAvalonTools
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

🔎 **코드 뜯어보기 (준비)** *(대부분 04·19에서 설명)*
- `import pickle`=저장된 모델 불러오기, `import time`=소요시간 측정.

### 셀 1 — 앙상블 모델 + 지문 계산기
21이 저장한 fingerprint 앙상블을 불러오고, **학습과 똑같은 순서**로 6종 지문을 만드는 함수를 정의한다.

In [ ]:
# 배포용 fingerprint 앙상블(21에서 저장) 불러오기 + 지문 계산기(학습과 동일 순서)
with open("data/HSD17B13_ensemble_screen_model.pkl", "rb") as f:
    B = pickle.load(f)
ens, fp_cols = B["ensemble"], B["fp_cols"]
NB = B["nbits"]
print("앙상블 로드 | top4:", B["top4"], "| 특징(지문) 수:", len(fp_cols))

gens = {
    "ecfp4":       rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NB),
    "rdkit":       rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NB),
    "atompair":    rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NB),
    "topotorsion": rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=NB),
}
def _bits(fp, n):
    a = np.zeros((n,), dtype=np.int8); DataStructs.ConvertToNumpyArray(fp, a); return a

def fp_matrix(mols):
    # 19와 '완전히 같은 순서'로 6종 지문을 이어붙여야 앙상블이 올바로 예측
    parts = [np.vstack([gens[k].GetFingerprintAsNumPy(m) for m in mols])
             for k in ("ecfp4", "rdkit", "atompair", "topotorsion")]
    parts.append(np.vstack([_bits(MACCSkeys.GenMACCSKeys(m), 167) for m in mols]))
    parts.append(np.vstack([_bits(pyAvalonTools.GetAvalonFP(m, NB), NB) for m in mols]))
    return np.hstack(parts).astype(np.float32)

🔎 **코드 뜯어보기 (셀 1)**
- `pickle.load(f)` : 저장된 딕셔너리(앙상블·지문열 순서 등) 복원. `ens.predict_proba`로 예측.
- `def fp_matrix(mols):` : 분자 목록을 6종 지문 행렬로. **19와 동일한 순서**(ecfp4→rdkit→atompair→topotorsion→maccs→avalon)로 `np.hstack`(좌우 결합)해야 앙상블이 올바로 해석. `.astype(np.float32)`=예측용 실수형.

### 셀 2 — 스크리닝할 3개 DB 목록
각 DB의 파일 경로·SMILES열·ID열·구분자를 지정한다.

In [ ]:
# 스크리닝할 3개 천연물 DB (파일·SMILES열·ID열·구분자)
LIBS = [
    ("FooDB",   "data/foodb_2020_4_7_csv/foodb_2020_04_07_csv/Compound.csv", "moldb_smiles",     "public_id", ","),
    ("NPASS",   "data/npass_structures.tsv",                                 "SMILES",           "np_id",     "\t"),
    ("COCONUT", "data/coconut_csv-09-2026.csv",                              "canonical_smiles", "identifier","," ),
]
PROB_MIN = 0.5      # 이 확률 이상만 저장(대부분은 무관물질이라 걸러짐)
CHUNK = 5000        # 한 번에 처리할 화합물 수(메모리 관리)

🔎 **코드 뜯어보기 (셀 2)**
- `LIBS = [(이름, 경로, smiles열, id열, 구분자), ...]` : 3개 DB 정보를 튜플 리스트로. `PROB_MIN=0.5`=이 확률 이상만 저장(대부분 무관물질은 버림). `CHUNK=5000`=한 번에 읽을 개수(메모리 관리).

### 셀 3 — 청크 단위 스크리닝
큰 파일을 5000개씩 나눠 읽어 지문→예측하고, 확률 높은 후보만 모은다.

In [ ]:
# 각 DB를 청크로 읽어 지문→앙상블 예측. prob>=PROB_MIN 후보만 모음.
def screen(name, path, smi_col, id_col, sep):
    hits, n_seen, t0 = [], 0, time.time()
    for chunk in pd.read_csv(path, sep=sep, usecols=[smi_col, id_col],
                             chunksize=CHUNK, on_bad_lines="skip", low_memory=False):
        ids, mols = [], []
        for cid, smi in zip(chunk[id_col], chunk[smi_col]):
            m = Chem.MolFromSmiles(str(smi))
            if m is not None:
                ids.append(cid); mols.append(m)
        n_seen += len(chunk)
        if not mols:
            continue
        X = fp_matrix(mols)
        prob = ens.predict_proba(X)[:, 1]
        for cid, m, p in zip(ids, mols, prob):
            if p >= PROB_MIN:
                hits.append((name, cid, Chem.MolToSmiles(m), float(p)))
        if n_seen % (CHUNK * 10) == 0:
            print(f"  [{name}] {n_seen} 처리, hit {len(hits)} ({time.time()-t0:.0f}s)")
    print(f"[{name}] 완료: {n_seen} 중 prob>={PROB_MIN} 후보 {len(hits)}개 ({time.time()-t0:.0f}s)")
    return hits

all_hits = []
for name, path, sc, ic, sep in LIBS:
    if not os.path.exists(path):
        print(f"[{name}] 파일 없음 → 건너뜀:", path); continue
    all_hits += screen(name, path, sc, ic, sep)

🔎 **코드 뜯어보기 (셀 3)**
- `pd.read_csv(path, chunksize=CHUNK)` : 파일을 **덩어리(chunk)씩** 읽는 반복자(700만 줄도 메모리 안전).
- `Chem.MolFromSmiles(...)` 로 파싱(실패 스킵). `ens.predict_proba(X)[:, 1]` : 청크 전체를 **한 번에** 예측.
- `if p >= PROB_MIN:` 인 것만 `hits`에 저장 → 대부분 버려 메모리 절약. DB 3개를 순서대로 처리(FooDB→NPASS→COCONUT).

### 셀 4 — 결과 정리·저장
모든 후보를 확률순 정렬해 CSV·Excel로 저장하고 DB별 개수·상위를 출력한다.

In [ ]:
# 결과 정리·저장: 전체 후보 + DB별 상위
res = pd.DataFrame(all_hits, columns=["source", "id", "canonical_smiles", "active_prob"])
res = res.sort_values("active_prob", ascending=False).reset_index(drop=True)
res.to_csv("data/screen_3db_hits.csv", index=False)
print("\n총 후보(prob>=%.2f): %d개" % (PROB_MIN, len(res)))
print(res.groupby("source").size().to_string())
print("\n=== 전체 상위 15 ===")
print(res.head(15).to_string(index=False))
with pd.ExcelWriter("data/screen_3db_hits.xlsx", engine="openpyxl") as w:
    res.head(1000).to_excel(w, sheet_name="top1000", index=False)
    for s in res["source"].unique():
        res[res.source == s].head(300).to_excel(w, sheet_name=f"{s}_top300", index=False)
print("저장: data/screen_3db_hits.csv , data/screen_3db_hits.xlsx")

🔎 **코드 뜯어보기 (셀 4)**
- `pd.DataFrame(all_hits, columns=[...])` : 후보 리스트를 표로. `.groupby("source").size()`=DB별 후보 수. `ExcelWriter`로 전체 top1000 + DB별 top300 시트 저장.